# BEAM CORE Workflow Demo: Provenance Tracking with Consist

This notebook is a small, self-contained version of a transportation modeling workflow: land use produces zone attributes, demand turns those attributes into trips, and network assignment turns trips into performance metrics. The model math is intentionally simple so the notebook can focus on what Consist adds around the workflow.

The important idea is that Consist tracks runs, inputs, outputs, configuration, cache identity, and lineage without forcing the model functions themselves to know about provenance. You can read this notebook after only skimming the docs; each section introduces the Consist API surface it uses before showing the code.

**What this notebook shows:**
- How to instrument a multi-step workflow with minimal changes around existing model code
- How scenario runs group related model steps under one named experiment
- How to compare two scenarios and see exactly what changed
- How cache reuse skips only the unchanged parts of a changed scenario
- How to trace an output back through the configuration and artifacts that produced it
- How to stage inputs for path-bound external tools without losing artifact identity

## Questions this notebook answers

- "What exact parameters produced this VMT forecast I sent to the MPO six months ago?"
- "Did we run the 2045 scenario with the updated BPR parameters or the old ones?"
- "Which trip table fed the network model in this scenario?"
- "Can a wrapper give a legacy tool the file path it expects while Consist still tracks the real source input?"

## Imports and Setup

The tracker is the object that records provenance. It writes run metadata and artifact links to a DuckDB database, and it uses `RUN_DIR` as the local workspace for this demo session.

A fixed `SESSION_ID` keeps the checked-in notebook outputs stable. In a real exploratory notebook you could use a timestamp or a meaningful scenario label instead.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from pydantic import BaseModel

import consist
from consist import CacheOptions, ExecutionOptions, Tracker


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repo root (missing pyproject.toml)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXAMPLES_DIR = REPO_ROOT / "examples"
RUN_DIR = EXAMPLES_DIR / "runs" / "beam_core_demo"
SESSION_ID = os.getenv("CONSIST_SESSION_ID", "demo")
DB_PATH = RUN_DIR / f"beam_core_demo_{SESSION_ID}.duckdb"

RUN_DIR.mkdir(parents=True, exist_ok=True)
if DB_PATH.exists():
    DB_PATH.unlink()

tracker = Tracker(
    run_dir=RUN_DIR,
    db_path=DB_PATH,
    hashing_strategy="fast",
    project_root=str(RUN_DIR),
)

print(f"Using tracker DB: {DB_PATH.relative_to(REPO_ROOT)}")

Using tracker DB: examples/runs/beam_core_demo/beam_core_demo_demo.duckdb


## Configuration

Each model has a tiny Pydantic config object. Consist also accepts plain dictionaries for `config=...`, but typed config objects are nicer for examples and real wrappers because they keep defaults, validation, and field names close to the code while still serializing cleanly for provenance.

The same objects are also passed into the Python model functions at runtime. That keeps the model code typed and readable while Consist handles serialization for cache identity, history, and diffs.


In [2]:
class LandUseConfig(BaseModel):
    n_zones: int = 5
    growth_rate: float = 0.02


class DemandConfig(BaseModel):
    value_of_time: float = 15.0
    transit_fare: float = 2.50
    parking_cost_cbd: float = 10.0


class NetworkConfig(BaseModel):
    capacity_factor: float = 1.0
    speed_reduction_factor: float = 0.5


land_use_config = LandUseConfig()
demand_config = DemandConfig()
network_config = NetworkConfig()

print(f"LandUseConfig: {land_use_config}")
print(f"DemandConfig: {demand_config}")
print(f"NetworkConfig: {network_config}")

LandUseConfig: n_zones=5 growth_rate=0.02
DemandConfig: value_of_time=15.0 transit_fare=2.5 parking_cost_cbd=10.0
NetworkConfig: capacity_factor=1.0 speed_reduction_factor=0.5


## Model Functions

These functions are deliberately ordinary Python. They accept inputs and return DataFrames, with no calls to Consist inside them.

That is the recommended integration shape for existing model code: keep domain logic clean, then add provenance at the workflow boundary. For a real external model, these functions might shell out to ActivitySim, BEAM, R, Java, or a container instead of doing the calculations inline.

In [3]:
def run_land_use(config: LandUseConfig) -> pd.DataFrame:
    base_data = [
        {"zone_id": 1, "zone_type": "suburban", "population": 5000, "employment": 1000},
        {"zone_id": 2, "zone_type": "inner", "population": 8000, "employment": 3000},
        {"zone_id": 3, "zone_type": "cbd", "population": 2000, "employment": 15000},
        {"zone_id": 4, "zone_type": "inner", "population": 7000, "employment": 4000},
        {"zone_id": 5, "zone_type": "suburban", "population": 4000, "employment": 800},
    ]
    if config.n_zones != 5:
        raise ValueError("This demo uses exactly 5 zones.")

    land_use = pd.DataFrame(base_data)
    growth_multiplier = 1.0 + config.growth_rate
    land_use["population"] = (land_use["population"] * growth_multiplier).round(0)
    land_use["employment"] = (land_use["employment"] * growth_multiplier).round(0)
    land_use["residential_units"] = (land_use["population"] / 2.3).round(0)
    return land_use


def _normalize_mode_shares(
    car: float, transit: float, walk: float
) -> tuple[float, float, float]:
    total = car + transit + walk
    if total <= 0:
        return 0.0, 0.0, 0.0
    return car / total, transit / total, walk / total


def run_demand_model(land_use: pd.DataFrame, config: DemandConfig) -> pd.DataFrame:
    rng = np.random.default_rng(42)
    zones = land_use[["zone_id", "zone_type", "population", "employment"]].copy()

    rows: list[dict[str, float | int | str]] = []
    for origin in zones.itertuples(index=False):
        for destination in zones.itertuples(index=False):
            if origin.zone_id == destination.zone_id:
                continue

            base_trips = (origin.population * destination.employment) / 100_000
            trip_noise = max(0.85, float(rng.normal(loc=1.0, scale=0.03)))
            base_trips *= trip_noise

            fare_penalty = max(0.0, config.transit_fare - 2.5) * 0.02
            vot_effect = (config.value_of_time - 15.0) * 0.002
            car_share = 0.60 + vot_effect
            transit_share = 0.25 - fare_penalty - (0.5 * vot_effect)
            walk_share = 0.15 + fare_penalty

            if destination.zone_type == "cbd":
                parking_shift = max(0.0, config.parking_cost_cbd - 10.0) * 0.015
                car_share -= parking_shift
                transit_share += parking_shift * 0.85
                walk_share += parking_shift * 0.15

            car_share, transit_share, walk_share = _normalize_mode_shares(
                max(car_share, 0.05),
                max(transit_share, 0.05),
                max(walk_share, 0.05),
            )

            distance_miles = abs(origin.zone_id - destination.zone_id) * 3.0
            for mode, share in (
                ("car", car_share),
                ("transit", transit_share),
                ("walk", walk_share),
            ):
                rows.append(
                    {
                        "origin_zone": int(origin.zone_id),
                        "dest_zone": int(destination.zone_id),
                        "mode": mode,
                        "trips": round(base_trips * share, 3),
                        "distance_miles": float(distance_miles),
                    }
                )

    return pd.DataFrame(rows)


def run_network_model(trip_table: pd.DataFrame, config: NetworkConfig) -> pd.DataFrame:
    car_trips = float(trip_table.loc[trip_table["mode"] == "car", "trips"].sum())
    transit_ridership = float(
        trip_table.loc[trip_table["mode"] == "transit", "trips"].sum()
    )
    total_trips = float(trip_table["trips"].sum())
    vmt = float(
        (
            trip_table.loc[trip_table["mode"] == "car", "trips"]
            * trip_table.loc[trip_table["mode"] == "car", "distance_miles"]
        ).sum()
    )

    capacity = 20_000.0 * config.capacity_factor
    volume_ratio = vmt / capacity if capacity > 0 else 0.0
    congestion_factor = 1.0 + 0.15 * (volume_ratio**4)

    free_flow_speed_mph = 35.0
    avg_network_speed_mph = free_flow_speed_mph / (
        1.0 + config.speed_reduction_factor * (congestion_factor - 1.0)
    )

    metrics = [
        {"metric": "vmt", "value": round(vmt, 3)},
        {"metric": "transit_ridership", "value": round(transit_ridership, 3)},
        {"metric": "total_trips", "value": round(total_trips, 3)},
        {"metric": "car_share", "value": round(car_trips / total_trips, 6)},
        {"metric": "avg_network_speed_mph", "value": round(avg_network_speed_mph, 3)},
        {"metric": "congestion_factor", "value": round(congestion_factor, 6)},
    ]
    return pd.DataFrame(metrics)

## Step Wrappers

The wrappers are the small provenance-aware layer around the model functions. Each wrapper calls the domain function, then logs the DataFrame under a stable artifact key such as `land_use` or `trip_table`.

Those keys matter. Downstream steps refer to them by name, lineage reports display them, and Consist can load or stage the corresponding artifact when a later run asks for it.

In [4]:
def land_use_step(config: LandUseConfig) -> None:
    land_use = run_land_use(config)
    consist.log_dataframe(land_use, key="land_use")


def demand_step(land_use: pd.DataFrame, config: DemandConfig) -> None:
    trip_table = run_demand_model(land_use, config)
    consist.log_dataframe(trip_table, key="trip_table")


def network_step(trip_table: pd.DataFrame, config: NetworkConfig) -> None:
    network_performance = run_network_model(trip_table, config)
    consist.log_dataframe(network_performance, key="network_performance")

## run_scenario() Helper

This helper is the main teaching cell in the notebook. It runs the three model steps under one scenario header so the baseline and policy cases are easy to compare.

A few details in the code are worth calling out:

- `consist.scenario(...)` creates the parent run that groups the land use, demand, and network child runs.
- `scenario.run(...)` executes one cache-aware step. If the step identity matches a previous completed run, Consist can return the cached outputs instead of calling the function again.
- `config=demand_config` is the Pydantic configuration object Consist hashes, stores, diffs, and shows in history.
- `runtime_kwargs={"config": demand_config}` passes that same object to the Python function at execution time. This keeps runtime argument binding explicit instead of relying on a magic config parameter convention.
- `inputs={"land_use": consist.ref(land_use_result, key="land_use")}` says that the demand step consumes the land-use artifact produced by the previous step.
- `input_binding="loaded"` tells Consist to load that artifact before calling the function, so `demand_step` receives a DataFrame rather than a filesystem path.

The result is a readable workflow definition where cache identity, runtime arguments, and artifact handoff are explicit.

In [5]:
def run_scenario(
    land_use_config: LandUseConfig,
    demand_config: DemandConfig,
    network_config: NetworkConfig,
    scenario_run_id: str,
) -> dict[str, object]:
    cache_options = CacheOptions(
        validate_cached_outputs="lazy",
        cache_hydration="inputs-missing",
    )

    with consist.scenario(
        scenario_run_id,
        tracker=tracker,
        config={
            "land_use": land_use_config.model_dump(),
            "demand": demand_config.model_dump(),
            "network": network_config.model_dump(),
        },
        facet_from=["land_use", "demand", "network"],
        tags=["demo", "beam_core"],
    ) as scenario:
        land_use_result = scenario.run(
            name="land_use_model",
            fn=land_use_step,
            config=land_use_config,
            outputs=["land_use"],
            execution_options=ExecutionOptions(
                runtime_kwargs={"config": land_use_config},
            ),
            cache_options=cache_options,
        )

        demand_result = scenario.run(
            name="demand_model",
            fn=demand_step,
            config=demand_config,
            inputs={"land_use": consist.ref(land_use_result, key="land_use")},
            outputs=["trip_table"],
            execution_options=ExecutionOptions(
                input_binding="loaded",
                runtime_kwargs={"config": demand_config},
            ),
            cache_options=cache_options,
        )

        network_result = scenario.run(
            name="network_model",
            fn=network_step,
            config=network_config,
            inputs={"trip_table": consist.ref(demand_result, key="trip_table")},
            outputs=["network_performance"],
            execution_options=ExecutionOptions(
                input_binding="loaded",
                runtime_kwargs={"config": network_config},
            ),
            cache_options=cache_options,
        )

    return {
        "scenario_run_id": scenario_run_id,
        "land_use_result": land_use_result,
        "demand_result": demand_result,
        "network_result": network_result,
    }

## Run Baseline Scenario

First run the baseline assumptions end-to-end. The helper returns the scenario run id and each step result, but most of the useful state is also persisted in the tracker database.

The next cell finds the baseline network step by parent scenario id and loads its `network_performance` output. This is the kind of lookup you need when a notebook, dashboard, or review memo starts from a scenario name rather than from an in-memory Python variable.

In [6]:
SCENARIO_NAME = "beam_core_demo"
base_run_id = f"{SCENARIO_NAME}_{SESSION_ID}_baseline"
baseline = run_scenario(land_use_config, demand_config, network_config, base_run_id)
BASELINE_RUN_ID = baseline["scenario_run_id"]
print("Baseline run complete.")

Baseline run complete.


In [7]:
network_run = tracker.find_run(
    parent_id=BASELINE_RUN_ID,
    model="network_model",
    status="completed",
)
if network_run is None:
    raise RuntimeError("Could not find baseline network_model run")

network_outputs = tracker.get_run_outputs(network_run.id)
consist.load_df(network_outputs["network_performance"], tracker=tracker)

,metric,value
0,vmt,15587.910000
1,transit_ridership,1366.224000
2,total_trips,5464.897000
3,car_share,0.600000
4,avg_network_speed_mph,34.057000
5,congestion_factor,1.055351


## Policy Scenario (Higher CBD Parking)

Now run the same workflow with a single policy change: CBD parking cost doubles from 10 to 20. The land-use and network configs are unchanged.

Because every step records its config and input artifacts, Consist can later distinguish the intentional policy change from everything that stayed constant.

In [8]:
high_parking_demand_config = DemandConfig(parking_cost_cbd=20.0)
policy_run_id = f"{SCENARIO_NAME}_{SESSION_ID}_high_parking"
policy = run_scenario(
    land_use_config,
    high_parking_demand_config,
    network_config,
    policy_run_id,
)
POLICY_RUN_ID = policy["scenario_run_id"]
print("Policy run complete.")

Policy run complete.


## What Did Consist Reuse?

The policy scenario is a more realistic caching example than the simple re-run in the quickstart notebook. We changed only the demand configuration, so Consist should reuse the unchanged land-use step, recompute demand, and then recompute network because its trip-table input changed.

This is stronger than a full identical rerun: it shows that the cache works at step granularity. Unrelated upstream work is skipped, while downstream work that depends on changed inputs is still invalidated.

In [9]:
policy_step_runs = tracker.find_runs(parent_id=POLICY_RUN_ID, status="completed")
policy_steps_by_model = {
    run.model_name: run
    for run in policy_step_runs
    if run.model_name in {"land_use_model", "demand_model", "network_model"}
}

reuse_rows = []
for model, reason in [
    ("land_use_model", "same land-use config and no upstream inputs changed"),
    ("demand_model", "parking cost changed in the demand config"),
    ("network_model", "trip-table input changed after demand recomputed"),
]:
    run = policy_steps_by_model[model]
    cache_hit = bool((run.meta or {}).get("cache_hit"))
    reuse_rows.append(
        {
            "model": model,
            "status": "cache_hit" if cache_hit else "recomputed",
            "reason": reason,
        }
    )

pd.DataFrame(reuse_rows)

,model,status,reason
0,land_use_model,cache_hit,same land-use config and no upstream inputs ch...
1,demand_model,recomputed,parking cost changed in the demand config
2,network_model,recomputed,trip-table input changed after demand recomputed


The next cell builds a small helper for loading scenario outputs from the provenance database. Given a scenario run id, it searches for the completed `network_model` child run, asks the tracker for that run's output artifacts, and hydrates the `network_performance` artifact back into a DataFrame.

That means the comparison does not depend on Python variables left over from the earlier cells. The run database is enough to rediscover the right artifact and load its bytes later, which is the same pattern you would use in a dashboard, report script, or follow-up analysis notebook.

In [10]:
def _load_network_performance(scenario_run_id: str) -> pd.DataFrame:
    run = tracker.find_run(
        parent_id=scenario_run_id,
        model="network_model",
        status="completed",
    )
    if run is None:
        raise RuntimeError(f"Could not find network_model run for {scenario_run_id}")
    outputs = tracker.get_run_outputs(run.id)
    return consist.load_df(outputs["network_performance"], tracker=tracker)


baseline_perf = _load_network_performance(BASELINE_RUN_ID).rename(
    columns={"value": "baseline"}
)
policy_perf = _load_network_performance(POLICY_RUN_ID).rename(
    columns={"value": "policy"}
)

comparison = baseline_perf.merge(policy_perf, on="metric", how="inner")
comparison["delta"] = (comparison["policy"] - comparison["baseline"]).round(3)

key_metrics = ["vmt", "transit_ridership", "car_share"]
comparison = comparison[comparison["metric"].isin(key_metrics)].copy()
comparison["metric"] = pd.Categorical(comparison["metric"], key_metrics, ordered=True)
comparison.sort_values("metric").reset_index(drop=True)

,metric,baseline,policy,delta
0,vmt,15587.910,13292.57400,-2295.336
1,transit_ridership,1366.224,1838.49600,472.272
2,car_share,0.600,0.49833,-0.102


## What Changed Between Scenarios?

Consist stores the full config for every run. `diff_runs()` compares two scenario headers and surfaces only the fields that actually changed.

This is useful because the scenario label alone is rarely enough evidence. A reviewer can see that this policy run differs in `demand.parking_cost_cbd`, not in land-use growth or network capacity.

In [11]:
diff = tracker.diff_runs(BASELINE_RUN_ID, POLICY_RUN_ID)

diff_rows = []
for key in sorted(diff["changes"]):
    change = diff["changes"][key]
    if change["status"] == "same":
        continue
    diff_rows.append(
        {
            "parameter": key,
            "baseline": change.get("left"),
            "policy": change.get("right"),
        }
    )

pd.DataFrame(diff_rows)

,parameter,baseline,policy
0,demand.parking_cost_cbd,10.0,20.0


## How Was This Output Produced?

Consist traces an output artifact back through its computation chain. The lineage tree shows the run that produced the artifact, the inputs that run consumed, and the upstream runs that produced those inputs.

For multi-model workflows this is often the fastest way to answer whether a result came from the expected demand table, network configuration, or scenario branch.

In [12]:
network_run = tracker.find_run(
    parent_id=BASELINE_RUN_ID,
    model="network_model",
    status="completed",
)
if network_run is None:
    raise RuntimeError("Could not find baseline network_model run")

network_art = tracker.get_run_outputs(network_run.id)["network_performance"]
tracker.print_lineage(network_art.id, max_depth=6)

Lineage
└── network_performance (parquet)
    └── ← network_model
        └── trip_table (parquet)
            └── ← demand_model
                └── land_use (parquet)
                    └── ← land_use_model

## Which Parameters Produced This VMT Number?

A common provenance question starts from a number, not from a run id. Here we load the baseline VMT value, keep the artifact id for the network output, then retrieve the demand-step config that fed it.

In a real review process, this is the difference between "I think this came from the baseline run" and a reproducible answer tied to a run id, artifact id, and stored configuration.

In [13]:
network_run = tracker.find_run(
    parent_id=BASELINE_RUN_ID,
    model="network_model",
    status="completed",
)
demand_run = tracker.find_run(
    parent_id=BASELINE_RUN_ID,
    model="demand_model",
    status="completed",
)
if network_run is None or demand_run is None:
    raise RuntimeError("Missing baseline demand/network runs")

network_outputs = tracker.get_run_outputs(network_run.id)
network_perf = consist.load_df(network_outputs["network_performance"], tracker=tracker)
vmt_value = network_perf.loc[network_perf["metric"] == "vmt", "value"].iloc[0]

demand_config_stored = tracker.get_run_config(demand_run.id)

pd.DataFrame(
    [
        {
            "vmt": vmt_value,
            "network_performance_artifact": str(
                network_outputs["network_performance"].id
            ),
            "demand_model_run": demand_run.id,
            **{
                key: demand_config_stored.get(key)
                for key in ["value_of_time", "transit_fare", "parking_cost_cbd"]
            },
        }
    ]
)

,vmt,network_performance_artifact,demand_model_run,value_of_time,transit_fare,parking_cost_cbd
0,15587.91,2b2ad6a1-1fe8-427d-b836-972dd6780808,beam_core_demo_demo_baseline_demand_model_5da6...,15.0,2.5,10.0


## Path-Bound Input Staging

The previous steps used `input_binding="loaded"` because they wanted DataFrames in memory. Many external tools work differently: they expect files at specific paths inside a workspace. A BEAM-style wrapper might need to provide `inputDirectory`, a legacy model might always read `workspace/config.txt`, or a container might expect mounted inputs under a fixed directory.

That creates a subtle provenance problem. If the wrapper manually copies files into a workspace, the tool can run, but the copied path can start to look like the source of truth. On cache hits, that workspace file may also be missing because the tool did not actually run.

Requested input staging solves that. The run declares the real source artifact in `inputs={...}`, then asks Consist to copy it to the tool-facing path with `ExecutionOptions(input_materialization="requested", input_paths={...})`. The source artifact remains the identity used for hashing and lineage; the staged path is just the runtime location handed to the tool.

In this toy example, `source_configs/legacy-demand-config.txt` is the real tracked input and `workspace/tool-config.txt` is the fixed filename the simulated legacy tool expects.

In [14]:
source_config_dir = RUN_DIR / "source_configs"
source_config_dir.mkdir(parents=True, exist_ok=True)
source_config_path = source_config_dir / "legacy-demand-config.txt"
source_config_path.write_text(
    "\n".join(
        [
            f"value_of_time={demand_config.value_of_time}",
            f"transit_fare={demand_config.transit_fare}",
            f"parking_cost_cbd={demand_config.parking_cost_cbd}",
        ]
    )
    + "\n",
    encoding="utf-8",
)

workspace_config_path = RUN_DIR / "workspace" / "tool-config.txt"


def path_bound_report_step(config_path: Path) -> dict[str, Path]:
    config = dict(
        line.split("=", 1)
        for line in config_path.read_text(encoding="utf-8").splitlines()
    )
    report_path = consist.output_path("path_bound_tool/report", ext="csv")
    report = pd.DataFrame(
        [
            {
                "staged_filename": config_path.name,
                "parking_cost_cbd": float(config["parking_cost_cbd"]),
                "status": "tool received fixed config path",
            }
        ]
    )
    report.to_csv(report_path, index=False)
    return {"tool_report": report_path}


staging_options = ExecutionOptions(
    input_binding="paths",
    input_materialization="requested",
    input_materialization_mode="copy",
    input_paths={"config_path": workspace_config_path},
)

staging_result = tracker.run(
    name="path_bound_report",
    fn=path_bound_report_step,
    model="legacy_path_bound_tool",
    config={"tool_contract": "fixed_config_path"},
    inputs={"config_path": source_config_path},
    outputs=["tool_report"],
    execution_options=staging_options,
)

print(f"Cache hit: {staging_result.cache_hit}")
print(f"Source identity path: {source_config_path.relative_to(RUN_DIR)}")
print(f"Tool saw staged path: {workspace_config_path.relative_to(RUN_DIR)}")
consist.load_df(staging_result.outputs["tool_report"], tracker=tracker)

Cache hit: False
Source identity path: source_configs/legacy-demand-config.txt
Tool saw staged path: workspace/tool-config.txt


,staged_filename,parking_cost_cbd,status
0,tool-config.txt,10.0,tool received fixed config path


The staging request is part of the run lifecycle, not an ad hoc copy before the run. That matters on cache hits: Consist can return the cached output while still restoring the fixed workspace input path for wrappers or downstream tools that expect it to exist.

The next cell deletes the staged file, reruns the same step, and shows both properties at once: the step is served from cache, and the staged input is recreated.

In [15]:
workspace_config_path.unlink(missing_ok=True)

staging_rerun = tracker.run(
    name="path_bound_report",
    fn=path_bound_report_step,
    model="legacy_path_bound_tool",
    config={"tool_contract": "fixed_config_path"},
    inputs={"config_path": source_config_path},
    outputs=["tool_report"],
    execution_options=staging_options,
)

print(f"Cache hit: {staging_rerun.cache_hit}")
print(f"Staged input restored: {workspace_config_path.exists()}")
workspace_config_path.read_text(encoding="utf-8")

Cache hit: True
Staged input restored: True


'value_of_time=15.0\ntransit_fare=2.5\nparking_cost_cbd=10.0\n'

## Summary

This workflow demonstrates four core Consist capabilities:

1. **Scenario diff**: `tracker.diff_runs()` shows exactly what changed between any two runs - no README archaeology, no folder-name decoding.
2. **Automatic lineage**: `tracker.print_lineage()` traces any output back through the full computation chain. Automatically maintained - no manual DAG management.
3. **Smart caching**: changed scenarios can reuse unaffected upstream steps, while config or input changes invalidate the affected downstream work.
4. **Path-bound staging**: `ExecutionOptions(input_materialization="requested", input_paths={...})` lets wrappers give external tools fixed local filenames without making those filenames the provenance identity.

These patterns scale from simple linear pipelines to sweeps and iterative workflows. Continue with `examples/02_parameter_sweep_monte_carlo.ipynb` for parameter sweeps and `examples/03_iterative_workflows.ipynb` for iterative workflows.

## Querying Run History

Every run is recorded in a queryable database. `tracker.history()` returns a DataFrame of all runs - IDs, timing, status, tags - without writing any additional logging code.

This is the same information that powers CLI inspection and later analysis. The important point is that provenance is not trapped inside this notebook session; it is stored with the tracker and can be queried later.

In [16]:
tracker.history()[["model_name", "status", "duration_seconds", "tags"]]

,model_name,status,duration_seconds,tags
0,legacy_path_bound_tool,completed,0.042753,[]
1,legacy_path_bound_tool,completed,0.091537,[]
2,network_model,completed,0.144172,[]
3,demand_model,completed,0.120355,[]
4,land_use_model,completed,0.027395,[]
5,scenario,completed,0.662626,"[demo, beam_core, scenario_header]"
6,network_model,completed,0.100647,[]
7,demand_model,completed,0.098754,[]
8,land_use_model,completed,0.101256,[]
9,scenario,completed,0.682147,"[demo, beam_core, scenario_header]"


The same run history is available from the command line - no notebook required:

![consist runs CLI](img/beam_core_demo_runs.png)